# ACS PUMS Data Ingestion

## Objective

This notebook ingests the **American Community Survey (ACS) Public Use Microdata Sample (PUMS)** datasets and prepares them for downstream analysis.

ACS PUMS provides individual- and household-level microdata describing the demographic, socioeconomic, and housing characteristics of the U.S. population. These records serve as the **structural foundation** for the audience segmentation framework developed in this project.

The raw data is distributed by the U.S. Census Bureau as large CSV files separated into two components:

- **HUS (Household files)** — housing and household-level characteristics  
- **PUS (Person files)** — demographic and socioeconomic attributes of individuals

Working directly with large CSV files is computationally inefficient. Therefore, this notebook performs an initial ingestion step that:

1. Loads the raw ACS PUMS CSV datasets
2. Verifies the presence of key sampling weights
3. Concatenates files when necessary
4. Converts the datasets to **Apache Parquet format** for efficient storage and processing

The resulting parquet datasets form the **starting point for structural feature engineering** in subsequent notebooks.

## Outputs

This notebook produces parquet versions of the ACS datasets stored in the project data directory, which will be used in later steps of the segmentation pipeline.

In [2]:
# Step 1 — Paths and file discovery

from pathlib import Path
import pandas as pd

# Project root (works when running notebooks from /notebooks)
PROJECT_ROOT = Path.cwd().parent

# Data directories
RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"

# ACS PUMS source directories
HUS_DIR = RAW_DIR / "acs" / "csv_hus"
PUS_DIR = RAW_DIR / "acs" / "csv_pus"

# Output directory for parquet datasets
OUT_DIR = INTERIM_DIR / "acs_pums_5y"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# File lists
hus_files = [HUS_DIR / f"psam_hus{x}.csv" for x in list("abcd")]
pus_files = [PUS_DIR / f"psam_pus{x}.csv" for x in list("abcd")]

# Validate files exist
for p in hus_files + pus_files:
    if not p.exists():
        raise FileNotFoundError(p)

print("HUS files:", [p.name for p in hus_files])
print("PUS files:", [p.name for p in pus_files])
print("Output directory:", OUT_DIR)

HUS files: ['psam_husa.csv', 'psam_husb.csv', 'psam_husc.csv', 'psam_husd.csv']
PUS files: ['psam_pusa.csv', 'psam_pusb.csv', 'psam_pusc.csv', 'psam_pusd.csv']
Output directory: /Users/marcomagnolo/Projects/us-audience-segmentation/data/interim/acs_pums_5y


In [2]:
# Step 2 - quick schema peek
def peek_csv(path, n=5):
    df = pd.read_csv(path, nrows=n)
    print(path.name, df.shape)
    return df

hus_peek = peek_csv(hus_files[0])
pus_peek = peek_csv(pus_files[0])

print("\nHUS columns:", len(hus_peek.columns))
print("PUS columns:", len(pus_peek.columns))

psam_husa.csv (5, 237)
psam_pusa.csv (5, 286)

HUS columns: 237
PUS columns: 286


In [3]:
# step 3 - confirms weights exist
def check_weight_cols(df, expected):
    missing = [c for c in expected if c not in df.columns]
    if missing:
        raise ValueError(f"Missing expected columns: {missing}")
    return True

check_weight_cols(hus_peek, ["WGTP"])
check_weight_cols(pus_peek, ["PWGTP"])

print("✅ Weight columns found: WGTP (HUS), PWGTP (PUS)")

✅ Weight columns found: WGTP (HUS), PWGTP (PUS)


In [4]:
# concatenate all HUS and PUS files
def load_concat_csv(paths):
    dfs = []
    for p in paths:
        df = pd.read_csv(p, low_memory=False)
        dfs.append(df)
        print(f"Loaded {p.name}: {df.shape}")
    out = pd.concat(dfs, ignore_index=True)
    print("Concatenated:", out.shape)
    return out

hus = load_concat_csv(hus_files)
pus = load_concat_csv(pus_files)

Loaded psam_husa.csv: (2199353, 237)
Loaded psam_husb.csv: (1686845, 237)
Loaded psam_husc.csv: (1823245, 237)
Loaded psam_husd.csv: (1940716, 237)
Concatenated: (7650159, 237)
Loaded psam_pusa.csv: (4695774, 286)
Loaded psam_pusb.csv: (3435956, 286)
Loaded psam_pusc.csv: (3714973, 286)
Loaded psam_pusd.csv: (4065690, 286)


: 

In [1]:
from pathlib import Path

PROJECT_ROOT = Path("/Users/marcomagnolo/Projects/Market_Kinetics")

HUS_DIR = PROJECT_ROOT / "data" / "societal_raw" / "ACS_US_Census" / "csv_hus"
PUS_DIR = PROJECT_ROOT / "data" / "societal_raw" / "ACS_US_Census" / "csv_pus"

OUT_DIR = PROJECT_ROOT / "data" / "societal_interim" / "acs_pums_5y"
OUT_DIR.mkdir(parents=True, exist_ok=True)

hus_files = [HUS_DIR / f"psam_hus{x}.csv" for x in "abcd"]
pus_files = [PUS_DIR / f"psam_pus{x}.csv" for x in "abcd"]

print("HUS:", [p.name for p in hus_files])
print("PUS:", [p.name for p in pus_files])
print("OUT:", OUT_DIR)
print("All HUS exist?", all(p.exists() for p in hus_files))
print("All PUS exist?", all(p.exists() for p in pus_files))

HUS: ['psam_husa.csv', 'psam_husb.csv', 'psam_husc.csv', 'psam_husd.csv']
PUS: ['psam_pusa.csv', 'psam_pusb.csv', 'psam_pusc.csv', 'psam_pusd.csv']
OUT: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y
All HUS exist? True
All PUS exist? True


In [ ]:
# fixing the code to load and concatenate the CSV files. first let's write HYS-A to PARQUET
p = hus_files[0]  # psam_husa.csv
out = OUT_DIR / f"{p.stem}.parquet"  # psam_husa.parquet

df = pd.read_csv(p, low_memory=False)
print("Loaded:", p.name, df.shape)

df.to_parquet(out, index=False, compression="zstd")
print("Wrote:", out)
del df

Loaded: psam_husa.csv (2199353, 237)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husa.parquet


In [4]:
# now let's do the same for all HUS files

p = hus_files[1]  # psam_husb.csv
out = OUT_DIR / f"{p.stem}.parquet"  # psam_husb.parquet

df = pd.read_csv(p, low_memory=False)
print("Loaded:", p.name, df.shape)

df.to_parquet(out, index=False, compression="zstd")
print("Wrote:", out)
del df

Loaded: psam_husb.csv (1686845, 237)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husb.parquet


In [ ]:
# now let's do the same for all HUS files

p = hus_files[2]  # psam_husc.csv
out = OUT_DIR / f"{p.stem}.parquet"

df = pd.read_csv(p, low_memory=False)
print("Loaded:", p.name, df.shape)

df.to_parquet(out, index=False, compression="zstd")
print("Wrote:", out)
del df

Loaded: psam_husc.csv (1823245, 237)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husc.parquet


Loaded: psam_husc.csv (1823245, 237)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husc.parquet


In [9]:
# now let's do the same for all HUS files

p = hus_files[3]  # psam_husc.csv
out = OUT_DIR / f"{p.stem}.parquet"

df = pd.read_csv(p, low_memory=False)
print("Loaded:", p.name, df.shape)

df.to_parquet(out, index=False, compression="zstd")
print("Wrote:", out)
del df

Loaded: psam_husd.csv (1940716, 237)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_husd.parquet


In [ ]:
PROJECT_ROOT = Path("/Users/marcomagnolo/Projects/Market_Kinetics")
PUS_DIR = PROJECT_ROOT / "data" / "societal_raw" / "ACS_US_Census" / "csv_pus"
OUT_DIR = PROJECT_ROOT / "data" / "societal_interim" / "acs_pums_5y"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pus_a = PUS_DIR / "psam_pusa.csv"
pus_a_out = OUT_DIR / "psam_pusa.parquet"

print("Input:", pus_a)
print("Output:", pus_a_out)
print("Exists input?", pus_a.exists())

Input: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_raw/ACS_US_Census/csv_pus/psam_pusa.csv
Output: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusa.parquet
Exists input? True


In [4]:
# load pus_a and convert to parquet

PROJECT_ROOT = Path("/Users/marcomagnolo/Projects/Market_Kinetics")
PUS_DIR = PROJECT_ROOT / "data" / "societal_raw" / "ACS_US_Census" / "csv_pus"
OUT_DIR = PROJECT_ROOT / "data" / "societal_interim" / "acs_pums_5y"
OUT_DIR.mkdir(parents=True, exist_ok=True)

pus_a = PUS_DIR / "psam_pusa.csv"
pus_a_out = OUT_DIR / "psam_pusa.parquet"

df = pd.read_csv(pus_a, low_memory=False)
print("Loaded:", pus_a.name, df.shape)

df.to_parquet(pus_a_out, index=False, compression="zstd")
print("Wrote:", pus_a_out)

del df

Loaded: psam_pusa.csv (4695774, 286)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusa.parquet


In [5]:
# now let's do the same for all PUS files

pus_b = PUS_DIR / "psam_pusb.csv"
pus_b_out = OUT_DIR / "psam_pusb.parquet"

df = pd.read_csv(pus_b, low_memory=False)
print("Loaded:", pus_b.name, df.shape)

df.to_parquet(pus_b_out, index=False, compression="zstd")
print("Wrote:", pus_b_out)

del df

Loaded: psam_pusb.csv (3435956, 286)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusb.parquet


In [6]:
# now let's do the same for all PUS files

pus_c = PUS_DIR / "psam_pusc.csv"
pus_c_out = OUT_DIR / "psam_pusc.parquet"

df = pd.read_csv(pus_c, low_memory=False)
print("Loaded:", pus_c.name, df.shape)

df.to_parquet(pus_c_out, index=False, compression="zstd")
print("Wrote:", pus_c_out)

del df

Loaded: psam_pusc.csv (3714973, 286)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusc.parquet


In [7]:
# now let's do the same for all PUS files

pus_d = PUS_DIR / "psam_pusd.csv"
pus_d_out = OUT_DIR / "psam_pusd.parquet"

df = pd.read_csv(pus_d, low_memory=False)
print("Loaded:", pus_d.name, df.shape)

df.to_parquet(pus_d_out, index=False, compression="zstd")
print("Wrote:", pus_d_out)

del df

Loaded: psam_pusd.csv (4065690, 286)
Wrote: /Users/marcomagnolo/Projects/Market_Kinetics/data/societal_interim/acs_pums_5y/psam_pusd.parquet
